In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task1')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task1/data/IT.csv')
oot = pl.read_csv('task1/data/OOT.csv')

num_cols = [c for c in it.columns if c.startswith('NUMERICAL_')]
it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')

X_train = train.select(num_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val.select(num_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot.select(num_cols).to_pandas()

In [ ]:
bp_all = BinningProcess(
    variable_names=num_cols,
    min_prebin_size=0.02, min_bin_size=0.05, max_n_bins=10,
)
bp_all.fit(X_train.values, y_train)

X_tr_woe_all = bp_all.transform(X_train.values, metric='woe')
X_va_woe_all = bp_all.transform(X_val.values, metric='woe')
X_oo_woe_all = bp_all.transform(X_oot.values, metric='woe')
print(f'WOE all features shape: {X_tr_woe_all.shape}')

for C in [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe_all, y_train)
    g = 2 * roc_auc_score(y_val, lr.predict_proba(X_va_woe_all)[:, 1]) - 1
    print(f'WOE_ALL+LR C={C}: val_gini={g:.4f}')

In [ ]:
bp_sel = BinningProcess(
    variable_names=num_cols,
    min_prebin_size=0.02, min_bin_size=0.05, max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp_sel.fit(X_train.values, y_train)
selected = list(bp_sel.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

X_tr_woe_sel = bp_sel.transform(X_train.values, metric='woe')
X_va_woe_sel = bp_sel.transform(X_val.values, metric='woe')
X_oo_woe_sel = bp_sel.transform(X_oot.values, metric='woe')

lr_sel = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr_sel.fit(X_tr_woe_sel, y_train)
p_lr_sel = lr_sel.predict_proba(X_va_woe_sel)[:, 1]
g_lr_sel = 2 * roc_auc_score(y_val, p_lr_sel) - 1
print(f'WOE_SEL+LR: val_gini={g_lr_sel:.4f}')

In [ ]:
configs = [
    {'num_leaves': 7, 'learning_rate': 0.01, 'min_child_samples': 500, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 15, 'learning_rate': 0.01, 'min_child_samples': 300, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 15, 'learning_rate': 0.005, 'min_child_samples': 500, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'bagging_freq': 3, 'reg_alpha': 3.0, 'reg_lambda': 3.0},
    {'num_leaves': 31, 'learning_rate': 0.01, 'min_child_samples': 200, 'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.5, 'reg_lambda': 0.5},
]

best_lgb_gini = 0
best_lgb_model = None
best_lgb_cfg = None

for i, cfg in enumerate(configs):
    params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'feature_pre_filter': False, **cfg}
    lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False)
    lgb_va = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)
    m = lgb.train(params, lgb_tr, num_boost_round=3000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    p = m.predict(X_val)
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'Config {i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]}, iters={m.best_iteration})')
    if g > best_lgb_gini:
        best_lgb_gini, best_lgb_model, best_lgb_cfg = g, m, cfg

p_lgb_val = best_lgb_model.predict(X_val)
print(f'\nBest LGB: val_gini={best_lgb_gini:.4f}')

In [ ]:
best_ens_g = 0
best_w = 0
for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_sel + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': g_lr_sel, 'LGB': best_lgb_gini, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE_LR':
    p_oot_final = lr_sel.predict_proba(X_oo_woe_sel)[:, 1]
elif best_name == 'LGB':
    p_oot_final = best_lgb_model.predict(X_oot)
else:
    p_oot_final = best_w * lr_sel.predict_proba(X_oo_woe_sel)[:, 1] + (1 - best_w) * best_lgb_model.predict(X_oot)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task1/predictions.csv')
print(f'OOT predictions: {preds.shape}')

with mlflow.start_run(run_name='v4_stability_focus'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features', len(num_cols))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.set_tag('task', 'task1')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')